In [201]:
'''
Submitted by: Criselvi lindsey
Class: DSC540 WK5&6 Assignment
Project Milestone 2
'''

'\nSubmitted by: Criselvi lindsey\nClass: DSC540 WK5&6 Assignment\nProject Milestone 2\n'

In [199]:
# Load the necessary libraries
import pandas as pd
import PyPDF2
import pdfplumber
import csv
import re
import seaborn as sns
import numpy as np
import openpyxl
import os
import requests
import json
from bs4 import BeautifulSoup

### Alameda County Labor Force.xls (Excel)

In [193]:
# 1. Load the Excel file, starting from the 14th row
file_path = 'Alameda County Labor Force.xls'
df = pd.read_excel(file_path, skiprows=12)

# 2. Clean up any unwanted columns by dropping completely empty columns
df = df.dropna(axis=1, how='all')

# Initialize an empty list to store year values
years = []

# 3. Detect the year and loop to populate an additional column for year
# Loop through the DataFrame to detect the year (the year is repeated in rows)
current_year = None
for index, row in df.iterrows():
    if isinstance(row.iloc[0], (int, float)):  # Check if the first column is a year
        current_year = row.iloc[0]
    years.append(current_year)

# 4. Add the 'Year' column to the DataFrame
df['Year'] = years

# 5. Clean up any unwanted columns by dropping completely empty columns
df = df.dropna(axis=1, how='all')

# 6. Remove the first row (which contains the incorrect header)
data_cleaned = df.drop(index=0)

# 7. Set the first row as the new header
data_cleaned.columns = data_cleaned.iloc[0]

# 8. Remove the first row (index 0)
df = data_cleaned.iloc[1:].reset_index(drop=True)

# 9. Drop rows where all values are NaN (null)
df = df.dropna(how='all', axis=0)  # Drop rows where all values are NaN

# 10.  Drop rows where any cell contains the phrase "Annual Averages"
df = df[~df.apply(lambda row: row.astype(str).str.contains('Annual Averages', case=False, na=False).any(), axis=1)]

# 11. Drop rows where any cell contains the phrase "Labor Force" (subsequent headers)
df = df[~df.apply(lambda row: row.astype(str).str.contains('Labor Force', case=False, na=False).any(), axis=1)]

# 12. Remove all rows with any NaN values (null values in any column)
df = df.dropna(how='any', axis=0)

# 13. Clean up any unwanted columns by  dropping completely empty columns again if needed
df = df.dropna(axis=1, how='all')

# 14. Rename the first column to 'Month' and the last column to 'Year'
df.columns.values[0] = 'Month'  # Rename the first column
df.columns.values[-1] = 'Year'  # Rename the last column

# 15. Convert the 'Year' column to an integer type (whole number)
df['Year'] = df['Year'].astype(int)

# 16. Save the cleaned DataFrame to a new CSV file
output_file = 'Cleaned_Alameda_County_Labor_Force.csv'
df.to_csv(output_file, index=False)

# Print the cleaned data (for preview)
print(df)

1        Month Labor Force Employment Unemployment Unemp. Rate  Year
0      January      866300     826200        40100      0.0462  2025
1     February      868900     830400        38500      0.0443  2025
15     January      863300     824600        38700      0.0448  2024
16    February      867800     829500        38300      0.0441  2024
17       March      870100     832900        37100      0.0426  2024
..         ...         ...        ...          ...         ...   ...
142     August      881200     842600        38600      0.0438  2016
143  September      884500     847900        36600      0.0413  2016
144    October      884900     848600        36300       0.041  2016
145   November      883500     849900        33500      0.0379  2016
146   December      880700     848400        32300      0.0366  2016

[110 rows x 6 columns]


 ###           California Urban EntireCCPI.pdf (PDF)

In [194]:
# 1. Function to extract table from PDF using pdfplumber
def extract_table_from_pdf(pdf_path, header_rows_to_skip=0):
    with pdfplumber.open(pdf_path) as pdf:
        all_tables = []
        for page in pdf.pages:
            # 2. Extract table from each page
            table = page.extract_table()
            if table:
                # 3. Remove the first 'header_rows_to_skip' number of rows
                if len(table) > header_rows_to_skip:
                    table = table[header_rows_to_skip:]  # Skip the header rows

                # 4. Clean up the column headers by removing spaces
                if table:
                    # 5. Assuming the first row is the header, update the header row
                    headers = table[0]
                    cleaned_headers = [header.replace(" ", "") for header in headers]  # Remove spaces
                    table[0] = cleaned_headers  # Update the header row

                all_tables.extend(table)  # Add the cleaned table data
        return all_tables


# 6. Function to write data to CSV
def write_to_csv(data, csv_path):
    with open(csv_path, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerows(data)

# Main process
pdf_file_path = "California Urban EntireCCPI.pdf" #Input file
csv_file_path = "Cleaned California_Urban_EntireCCPI.csv"  # Output CSV file path

# Define the number of header rows to skip
header_rows_to_skip = 0  # For example, skip the first 3 header rows

# Extract table from the PDF while skipping header rows
table_data = extract_table_from_pdf(pdf_file_path, header_rows_to_skip)

# Check if the table data is not empty before processing
if table_data:
    # Convert the extracted table data into a pandas DataFrame
    df_table_data = pd.DataFrame(table_data[1:], columns=table_data[0])  # Set the first row as headers

    # Drop rows where any column contains the value 'AllUrbanConsumers'
    df_table_data = df_table_data[~df_table_data.apply(lambda row: row.astype(str).str.contains('AllUrbanConsumers').any(), axis=1)]

    # Print the first 5 rows of the DataFrame after filtering
    print("First 5 rows of cleaned data:")
    print(df_table_data.head())

    # Write the cleaned table data into a CSV file
    write_to_csv(df_table_data.values.tolist(), csv_file_path)
    print(f"Data has been extracted, cleaned, and written to {csv_file_path}")
else:
    print("No table data found in the PDF.")


First 5 rows of cleaned data:
   Year     Month AllUrbanConsumers UrbanWageEarners\nandClericalWorkers
0  2024    Annual           341.951                              330.689
1  2024  December           344.295                              332.708
2  2024   October           343.575                              332.293
3  2024    August           343.108                              331.734
4  2024      June           342.856                              331.398
Data has been extracted, cleaned, and written to Cleaned California_Urban_EntireCCPI.csv


###           Historical_Income.csv (HTML)

In [196]:
# Import California Average Income Historical Data
# The full URL to the data file
url = "https://fred.stlouisfed.org/data/MEHOINUSCAA672N.txt"

# Send the request to download the data
response = requests.get(url)

# Save the content to a local file
with open("MEHOINUSCAA672N.txt", "wb") as file:
    file.write(response.content)

print("File downloaded successfully.")


File downloaded successfully.


In [197]:
# Load the downloaded historical income file
with open("MEHOINUSCAA672N.txt", "r", encoding="utf-8") as file:
    html_content = file.read()

# 1. Parse the HTML content
soup = BeautifulSoup(html_content, 'html.parser')

# 2. Find the table with the specific ID
table = soup.find('table', id='data-table-observations')

# 3. Check if the table exists in the HTML
if table:
    # Find all rows within the specific table
    rows = table.find_all('tr')

    # 4. Prepare a list to store data
    data = []

    # 5. Print the column headers
    print(f"{'Year':<10}{'AvgIncome'}") 

    # 6. Loop through each row and extract the date (year) and value (average income)
    for row in rows:
        date_cell = row.find('th', scope='row')  # Find the <th> with the date
        value_cell = row.find('td')  # Find the <td> with the value

        if date_cell and value_cell:
            date = date_cell.text.strip()  # Get the date text
            value = value_cell.text.strip()  # Get the value text

            # Extract year from date
            year = int(date.split('-')[0])

            # 7. Only include data from 2020 onwards
            if year >= 2020:
                # Print the year and AvgIncome
                print(f"{year:<10}{value}")

                # Store the data in the list
                data.append([year, value])

    # 8. Save the extracted data into a CSV file
    with open('Cleaned_California_Avg_Income.csv', 'w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        # Write headers (column names) with "AvgIncome"
        writer.writerow(["Year", "AvgIncome"])
        # Write data rows
        writer.writerows(data)

    print("\nData from 2020 onwards has been saved to 'Cleaned_California_Avg_Income.csv'.")
else:
    print("Table with id 'data-table-observations' not found.")


Year      AvgIncome
2020      90840
2021      91350
2022      88690
2023      89870

Data from 2020 onwards has been saved to 'Cleaned_California_Avg_Income.csv'.


###         Bay Area Housing Data (API)

In [109]:
# Define the URL and base parameters
url = 'https://api.rentcast.io/v1/listings/sale'
params = {
    'latitude': 37.6688,
    'longitude': -122.0810,
    'radius': 10,
    'limit': 500,  # Set the limit of records per page
    'offset': 0     # Start from the first page (offset = 0)
}

# Define the headers and use the API key. This is a paid API subscribtion, so I am removing the key after running the API call.
headers = {
    'Accept': 'application/json',
    'X-Api-Key': 'YOUR_RENTCAST_API_KEY'
}

# Create an empty list to store all the data
all_data = []

# Set the initial offset
offset = 0

# Loop to fetch all data, stopping when offset is >= 20000
while offset < 10001:
    # Update the offset in the parameters
    params['offset'] = offset
    
    # Send GET request to the API
    response = requests.get(url, headers=headers, params=params)
    
    # Check if the request was successful (status code 200)
    if response.status_code == 200:
        # Parse the JSON response into a dictionary
        data = response.json()
        
        # If there's no data in the response, break the loop
        if len(data) == 0:
            print("No more data available. Exiting...")
            break
        
        # Append the new data to the all_data list
        all_data.extend(data)  # Use extend to add all elements from the data

        # Increment the offset by 500 for the next API call (for the next page)
        offset += 500
        print(f"Fetching next page, new offset: {offset}")
    else:
        # Handle request failure
        print(f"Request failed with status code {response.status_code}")
        print("Response:", response.text)
        break

# If there's data, save it to a CSV file
if all_data:
    # Convert the list of dictionaries into a pandas DataFrame
    all_data_df = pd.json_normalize(all_data)
    
    # Save the combined data to a CSV file
    all_data_df.to_csv("all_property_data_until_20000_offset_04062025.csv", index=False)
    print("\nData has been saved to 'all_property_data_until_20000_offset_04062025.csv'.")


Fetching next page, new offset: 500
Fetching next page, new offset: 1000
Fetching next page, new offset: 1500
Fetching next page, new offset: 2000
Fetching next page, new offset: 2500
Fetching next page, new offset: 3000
Fetching next page, new offset: 3500
Fetching next page, new offset: 4000
Fetching next page, new offset: 4500
Fetching next page, new offset: 5000
Fetching next page, new offset: 5500
Fetching next page, new offset: 6000
Fetching next page, new offset: 6500
Fetching next page, new offset: 7000
Fetching next page, new offset: 7500
Fetching next page, new offset: 8000
Fetching next page, new offset: 8500
Fetching next page, new offset: 9000
Fetching next page, new offset: 9500
Fetching next page, new offset: 10000
Fetching next page, new offset: 10500

Data has been saved to 'all_property_data_until_20000_offset_04062025.csv'.


In [200]:
# 1. Load the saved CSV file into a DataFrame
file_path = "all_property_data_until_20000_offset_04062025.csv"

# 2. Specify dtypes, using 'Int64' (nullable integer) for columns that could have missing values
# Use 'Float64' for columns that can have decimal numbers
dtypes = {
    'zipCode': 'Int64',  # Nullable integer to handle missing values
    'price': 'Float64',   # Use 'Float64' for columns with decimal values
    'mlsNumber': 'str',   # Keep as string if it contains any non-numeric data
    'bedrooms': 'Float64',  # Nullable float to handle possible decimals
    'bathrooms': 'Float64', # Nullable float
    'squareFootage': 'Float64', # Nullable float
    'yearBuilt': 'Int64',     # Nullable integer
    'hoa.fee': 'Float64',       # Nullable float
    'listingType': 'str',     # String
    'daysOnMarket': 'Int64'   # Nullable integer
}


# 3. Specify the columns that should be parsed as dates
date_columns = ['listedDate', 'removedDate', 'createdDate', 'lastSeenDate']

# 4. Load the data with the specified dtypes, parse the date columns, and set low_memory=False
df = pd.read_csv(file_path, dtype=dtypes, parse_dates=date_columns, low_memory=False)

# 5. Since there are so many unwanted columns, find all columns that contain the word "history" (case-insensitive)
columns_to_remove_history = [col for col in df.columns if 'history' in col.lower()]

# 6. Find all columns that contain the word "listing" (case-insensitive), but exclude 'listingType'
columns_to_remove_listing = [col for col in df.columns if 'listing' in col.lower() and col != 'listingType']

# 7. Specify additional columns to remove (replace with actual column names)
additional_columns_to_remove = ['id', 'latitude', 'longitude', 'builder.name', 'builder.development', 'builder.phone']

# 8. Combine all the columns to remove (history, listing, and additional specified columns)
columns_to_remove = columns_to_remove_history + columns_to_remove_listing + additional_columns_to_remove

# 9. Drop the specified columns
df_cleaned = df.drop(columns=columns_to_remove)

# 10. Convert any columns with 'date' in the name to datetime if needed
date_columns = [col for col in df_cleaned.columns if 'date' in col.lower()]

# 11. Now convert to just date (removing time part)
for col in date_columns:
    if pd.api.types.is_datetime64_any_dtype(df_cleaned[col]):
        df_cleaned[col] = df_cleaned[col].dt.date  # Convert datetime to date (removes time)
    else:
        print(f"Warning: Column {col} was not converted to datetime.")

# 12. Remove the word "county" from the 'county' column
df_cleaned['county'] = df_cleaned['county'].str.replace('county', '', case=False, regex=True)

# 10. Filter the data for only 'Single Family' propertyType
df_cleaned = df_cleaned[df_cleaned['propertyType'] == 'Single Family']

# 13. Count before removing duplicates
before_count = df_cleaned.shape[0]
print(f"Number of rows before duplicate removal: {before_count}")

# 14. Print the duplicate records to a file verify against the actual file
duplicates = df_cleaned[df_cleaned.duplicated()]

# Save duplicates to a CSV file
duplicates.to_csv("duplicate_housing_records.csv", index=False)

print("Duplicate records have been saved to 'duplicate_housing_records.csv'.")

# 15. Remove any duplicate rows
df_cleaned = df_cleaned.drop_duplicates()

# 16. Count after removing duplicates
after_count = df_cleaned.shape[0]
print(f"Number of rows after duplicate removal: {after_count}")

# 17. Fill missing 'bedrooms' with the mode (most frequent value)
bedroom_mode = df_cleaned['bedrooms'].mode().iloc[0]  # mode() returns a Series, so take the first value
df_cleaned['bedrooms'] = df_cleaned['bedrooms'].fillna(bedroom_mode)

# 18. Fill missing 'bathrooms' with the mode (most frequent value)
bathroom_mode = df_cleaned['bathrooms'].mode().iloc[0]  # mode() returns a Series, so take the first value
df_cleaned['bathrooms'] = df_cleaned['bathrooms'].fillna(bedroom_mode)

# Save the cleaned DataFrame back to a new CSV file
df_cleaned.to_csv("cleaned_property_data.csv", index=False)

Number of rows before duplicate removal: 7193
Duplicate records have been saved to 'duplicate_housing_records.csv'.
Number of rows after duplicate removal: 7178


#################################################################################################
###                 Alameda County Labor Force.xls (Excel)
 In processing the Excel file (Alameda County Labor Force.xls) and PDF file (California Urban EntireCCPI.pdf),
 several transformations were made including the following:
             Alameda County Labor Force.xls
 1. Load the Excel file, starting from the 14th row
 2. Clean up any unwanted columns by dropping completely empty columns
 3. Detect the year and loop to populate an additional column for year
 4. Add the 'Year' column to the DataFrame
 5. Clean up any unwanted columns by dropping completely empty columns
 6. Remove the first row (which contains the incorrect header)
 7. Set the first row as the new header
 8. Remove the first row (index 0)
 9. Drop rows where all values are NaN (null)
 10. Drop rows where any cell contains the phrase "Annual Averages"
 11. Drop rows where any cell contains the phrase "Labor Force" (subsequent headers)
 12. Remove all rows with any NaN values (null values in any column)
 13. Clean up any unwanted columns by  dropping completely empty columns again if needed
 14. Rename the first column to 'Month' and the last column to 'Year'
 15. Convert the 'Year' column to an integer type (whole number)
 16. Save the cleaned DataFrame to a new CSV file

 ###           California Urban EntireCCPI.pdf (PDF)
 1. Function to extract table from PDF using pdfplumber
 2. Extract table from each page
 3. Remove the first 'header_rows_to_skip' number of rows
 4. Clean up the column headers by removing spaces
 5. Assuming the first row is the header, Update the header row
 6. Function to write data to CSV

 ###           Historical_Income.csv (HTML)
 1. Parse the HTML content
 2. Find the table with the specific ID
 3. Check if the table exists in the HTML
 4. Prepare a list to store data
 5. Print the column headers
 6. Loop through each row and extract the date (year) and value (average income)
 7. Only include data from 2020 onwards
 8. Save the extracted data into a CSV file

###         Bay Area Housing Data (API)
 1. Load the saved CSV file into a DataFrame
 2. Specify dtypes, using 'Int64' (nullable integer) for columns that could have missing values
 3. Specify the columns that should be parsed as dates
 4. Load the data with the specified dtypes, parse the date columns, and set low_memory=False
 5. Since there are so many unwanted columns, find all columns that contain the word "history" (case-insensitive)
 6. Find all columns that contain the word "listing" (case-insensitive), but exclude 'listingType'
 7. Specify additional columns to remove (replace with actual column names)
 8. Combine all the columns to remove (history, listing, and additional specified columns)
 9. Drop the specified columns
 10. Convert any columns with 'date' in the name to datetime if needed
 11. Now convert to just date (removing time part)
 12. Remove the word "county" from the 'county' column
 10. Filter the data for only 'Single Family' propertyType
 13. Count before removing duplicates
 14. Print the duplicate records to a file verify against the actual file
 15. Remove any duplicate rows
 16. Count after removing duplicates
 17. Fill missing 'bedrooms' with the mode (most frequent value)
 18. Fill missing 'bathrooms' with the mode (most frequent value)

 ### PARAGRAPH OF THE ETHICAL IMPLICATIONS OF THE DATA WRANGLING SPECIFIC TO THE DATASOURCE AND THE

In this project, data was wrangled from publicly available sources, including a PDF report from the California Consumer Price Index and a CSV file of historical income from a government archive. Changes included cleaning column headers (removing spaces and special characters), skipping redundant header rows, dropping repeated labels, and saving cleaned outputs to CSV for analysis. These transformations were necessary for readability and consistency but came with ethical considerations. While the data itself is public and thus legally usable, there is a responsibility to preserve its integrity. Risks include misinterpretation, especially if metadata or explanatory footnotes were removed. The data, sourced from public government documents, is assumed to be credible and legally available for public use, aligning with ethical and regulatory guidelines for open government data. However, assumptions were made in identifying years and distinguishing between data rows and header rows, which could lead to misclassification if the structure varied across pages or years. To mitigate the mentioned risks, it is important to document all data cleaning steps transparently, retain original copies for reference, and validate cleaned data against known statistics or benchmarks. Ensuring future users are aware of these transformations helps uphold both ethical responsibility and analytical.
